## Data Split ( Train, Val, Test)

In [ ]:

import os, shutil
from pathlib import Path
from typing import Sequence, Tuple, List
import torch
from torch.utils.data import random_split

def list_files(folder: Path, exts: Tuple[str, ...] = None) -> List[Path]:
    files = []
    for p in folder.iterdir():
        if p.is_file():
            if exts is None or p.suffix.lower() in exts:
                files.append(p)
    return sorted(files)

def ensure_dir(p: Path): p.mkdir(parents=True, exist_ok=True)

def copy_many(files: List[Path], dst_dir: Path, move: bool = False):
    ensure_dir(dst_dir)
    for f in files:
        dst = dst_dir / f.name
        if move: shutil.move(str(f), str(dst))
        else:    shutil.copy2(str(f), str(dst))

def split_one_class(
    source_dir: str,
    out_train: str, out_val: str, out_test: str,
    ratios: Sequence[float] = (0.7, 0.15, 0.15),
    seed: int = 42,
    move: bool = False,
    exts: Tuple[str, ...] = None
):
    src = Path(source_dir)
    files = list_files(src, exts)
    n = len(files)
    assert n > 0, f"No files in {src}"

    # lengths from ratios
    rsum = float(sum(ratios))
    ratios = [r/rsum for r in ratios]
    lengths = [int(r * n) for r in ratios]
    lengths[-1] = n - sum(lengths[:-1])  # fix rounding

    # deterministic split with torch.utils.data.random_split
    g = torch.Generator().manual_seed(seed)
    subsets = random_split(files, lengths, generator=g)  # returns Subset objects
    train_files = [files[i] for i in subsets[0].indices]
    val_files   = [files[i] for i in subsets[1].indices]
    test_files  = [files[i] for i in subsets[2].indices]

    copy_many(train_files, Path(out_train), move=move)
    copy_many(val_files,   Path(out_val),   move=move)
    copy_many(test_files,  Path(out_test),  move=move)

    mode = "MOVED" if move else "COPIED"
    print(f"[DONE] {mode} {n} files from {src.name} → "
          f"{len(train_files)}/{len(val_files)}/{len(test_files)} (train/val/test)")

if __name__ == "__main__":
    
    #  (Fight class) ---
    # SOURCE = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\Fight"
    # OUT_T  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\train\Fight"
    # OUT_V  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\val\Fight"
    # OUT_S  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\test\Fight"

    #  (NonFight class) ---
    SOURCE = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\NonFight"
    OUT_T  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\train\NonFight"
    OUT_V  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\val\NonFight"
    OUT_S  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRWF\test\NonFight"




    split_one_class(SOURCE, OUT_T, OUT_V, OUT_S,
                    ratios=(0.7, 0.15, 0.15),
                    seed=42, move=False, exts=None)  


[DONE] COPIED 4760 files from NonFight → 3332/714/714 (train/val/test)


# Feature extraction

In [ ]:

import os, math, cv2, numpy as np, torch
from ultralytics import YOLO
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.transforms import functional as TF

# ---------------- Config ----------------
# We will process ALL frames, but choose exactly 75 frames at the end
TGT_FRAMES    = 75
TOP_K         = 4
CONF_W        = 0.4
MOT_W         = 0.6
SMOOTH_ALPHA  = 0.2
YOLO_WEIGHTS  = "yolov8s-pose.pt"
ROI_TOPN      = 2
ROI_EXPAND    = 1.25
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_DEVICE   = 0 if DEVICE == "cuda" else DEVICE
USE_HALF      = torch.cuda.is_available()

# ---------------- Models ----------------
torch.backends.cudnn.benchmark = True
pose_model = YOLO(YOLO_WEIGHTS)

rgb_weights = MobileNet_V3_Large_Weights.DEFAULT
rgb_model   = mobilenet_v3_large(weights=rgb_weights).to(DEVICE).eval()
# Keep the first Linear(960->1280) + HSwish to get a 1280-D embedding
rgb_model.classifier = torch.nn.Sequential(
    rgb_model.classifier[0],
    rgb_model.classifier[1],
)
rgb_tf = rgb_weights.transforms()

# ---------------- Pose utils ----------------
COCO_WRISTS = [9,10]; COCO_HEAD = [0,1,2,3,4]

def extract_pose(results, W, H):
    """Return per-person 51-dim (17*(x,y,conf)) normalized, avg conf, and center."""
    if (not results) or (results[0].keypoints is None):
        return np.empty((0,51),np.float32), np.empty((0,),np.float32), np.empty((0,2),np.float32)
    feats, conf_avg, cents = [], [], []
    for kp in results[0].keypoints:
        xy = kp.xy.cpu().numpy().squeeze(0)      # (17,2)
        cf = kp.conf.cpu().numpy().squeeze(0)    # (17,)
        xy[:,0] /= max(W,1); xy[:,1] /= max(H,1)
        feats.append(np.hstack([xy, cf[:,None]]).astype(np.float32).flatten())  # (51,)
        conf_avg.append(cf.mean().astype(np.float32))
        cents.append(xy.mean(axis=0).astype(np.float32))
    return np.asarray(feats,np.float32), np.asarray(conf_avg,np.float32), np.asarray(cents,np.float32)

def match_prev_curr(prev_c, curr_c, max_dist=0.2):
    """Greedy matching to keep track-like stability across frames."""
    N,M = curr_c.shape[0], prev_c.shape[0]
    if N==0 or M==0: return [-1]*N
    d = np.linalg.norm(curr_c[:,None,:]-prev_c[None,:,:], axis=2)
    pairs = [(i,j,d[i,j]) for i in range(N) for j in range(M)]
    pairs.sort(key=lambda x:x[2])
    used=set(); out=[-1]*N
    for i,j,dd in pairs:
        if out[i]==-1 and j not in used and dd<=max_dist:
            out[i]=j; used.add(j)
    return out

def motion_score(cur51, prv51):
    """Average keypoint displacement magnitude."""
    xy_c = cur51.reshape(17,3)[:,:2]; xy_p = prv51.reshape(17,3)[:,:2]
    diff = xy_c - xy_p
    return float(np.sqrt((diff**2).sum(axis=1)).mean())

def add_velocity_camcomp(cur51, prv51, gdx=0.0, gdy=0.0):
    """Append velocities with simple camera-compensation to current 51-dim."""
    if prv51 is None:
        vel = np.zeros_like(cur51, np.float32)
    else:
        cur = cur51.reshape(17,3); prv = prv51.reshape(17,3)
        dv  = (cur - prv).astype(np.float32)
        dv[:,:2] -= np.array([gdx, gdy], dtype=np.float32)
        vel = dv.reshape(-1)
    return np.hstack([cur51, vel]).astype(np.float32)  # -> 102 dims

def bbox_from_kp(kp51):
    """Tight bbox from 17 normalized keypoints."""
    xy = kp51.reshape(17,3)[:,:2]
    x1,y1 = xy[:,0].min(), xy[:,1].min()
    x2,y2 = xy[:,0].max(), xy[:,1].max()
    return np.array([x1,y1,x2,y2], np.float32)

def iou_xyxy(a, b):
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw*ih
    ua = max(0.0, ax2-ax1)*max(0.0, ay2-ay1) + max(0.0, bx2-bx1)*max(0.0, by2-by1) - inter + 1e-9
    return float(inter/ua)

def signed_approach(ca, cb, pa=None, pb=None):
    """Positive if people are approaching each other (relative velocity projection)."""
    if pa is None or pb is None: return 0.0
    r = cb - ca; norm = np.linalg.norm(r) + 1e-9; r_hat = r / norm
    va = ca - pa; vb = cb - pb; vrel = vb - va
    return - float(np.dot(vrel, r_hat))

def min_wrist_head_dist(kpA51, kpB51):
    """Min distance from wrists of one to head of the other (contact proxy)."""
    A = kpA51.reshape(17,3)[:,:2]; B = kpB51.reshape(17,3)[:,:2]
    wristsA, headsB = A[COCO_WRISTS], B[COCO_HEAD]
    wristsB, headsA = B[COCO_WRISTS], A[COCO_HEAD]
    d1 = np.sqrt(((wristsA[:,None,:]-headsB[None,:,:])**2).sum(axis=2)).min()
    d2 = np.sqrt(((wristsB[:,None,:]-headsA[None,:,:])**2).sum(axis=2)).min()
    return float(min(d1, d2))

# ---------------- Flow utils ----------------
def flow_features(prev_gray, gray, prev_flow=None):
    """13-dim global flow descriptor + return full flow for FoF."""
    if prev_gray is None or gray is None:
        return np.zeros(13, np.float32), None, 0.0
    pg = cv2.resize(prev_gray, (0,0), fx=0.5, fy=0.5)
    g  = cv2.resize(gray,       (0,0), fx=0.5, fy=0.5)
    flow = cv2.calcOpticalFlowFarneback(pg, g, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    u, v = flow[...,0], flow[...,1]
    mag, ang = cv2.cartToPolar(u, v, angleInDegrees=False)
    mean_u, mean_v = float(u.mean()), float(v.mean())
    std_u, std_v   = float(u.std()),  float(v.std())
    mean_m, std_m  = float(mag.mean()), float(mag.std())
    mean_sin, mean_cos = float(np.sin(ang).mean()), float(np.cos(ang).mean())
    bins = np.linspace(0, 2*np.pi, 5)
    hist,_ = np.histogram(ang, bins=bins)
    hist = (hist / (hist.sum()+1e-9)).astype(np.float32)
    base12 = np.array([mean_u,mean_v,std_u,std_v,mean_m,std_m,mean_sin,mean_cos,*hist], np.float32)
    if prev_flow is None:
        fof = 0.0
    else:
        du = u - prev_flow[...,0]; dv = v - prev_flow[...,1]
        fof = float(np.sqrt(du*du + dv*dv).mean())
    return np.hstack([base12, [fof]]).astype(np.float32), flow, float(mean_m)

# ---------------- ROI helper ----------------
def union_roi_norm(boxes_xyxy, take_topn=2, expand=1.25):
    """Union ROI of top-N normalized boxes with expansion."""
    if len(boxes_xyxy) == 0:
        return np.array([0.0,0.0,1.0,1.0], np.float32)
    take = min(take_topn, len(boxes_xyxy))
    b = boxes_xyxy[:take]
    x1 = float(b[:,0].min()); y1 = float(b[:,1].min())
    x2 = float(b[:,2].max()); y2 = float(b[:,3].max())
    cx = (x1+x2)/2.0; cy=(y1+y2)/2.0; w=(x2-x1); h=(y2-y1)
    w *= expand; h *= expand
    x1 = np.clip(cx - w/2.0, 0.0, 1.0); y1 = np.clip(cy - h/2.0, 0.0, 1.0)
    x2 = np.clip(cx + w/2.0, 0.0, 1.0); y2 = np.clip(cy + h/2.0, 0.0, 1.0)
    if (x2-x1)<1e-4 or (y2-y1)<1e-4:
        return np.array([0.0,0.0,1.0,1.0], np.float32)
    return np.array([x1,y1,x2,y2], np.float32)

# ---------------- Score-aware frame selection ----------------
def pick_indices_score_aware(scores, T_target):
    """
    Choose exactly T_target indices from scores (len T_all).
    If all scores ~0, fall back to uniform sampling.
    Otherwise, select by equal-spaced points on cumulative score (importance sampling).
    """
    scores = np.asarray(scores, np.float32)
    scores = np.maximum(scores, 0)
    # light smoothing
    if len(scores) >= 5:
        k = 5
        kernel = np.ones(k, np.float32) / k
        scores = np.convolve(scores, kernel, mode='same')
    total = float(scores.sum())
    T_all = len(scores)
    if T_all <= T_target:
        return np.arange(T_all, dtype=np.int64)
    if total < 1e-8:
        # uniform fallback
        idx = np.linspace(0, T_all-1, T_target).round().astype(np.int64)
        return np.unique(idx)
    c = np.cumsum(scores)
    targets = np.linspace(c[0], c[-1], T_target+2)[1:-1]
    idx = np.searchsorted(c, targets)
    idx = np.clip(idx, 0, T_all-1).astype(np.int64)
    # ensure uniqueness; if not enough, fill with uniform picks
    idx = np.unique(idx)
    if len(idx) < T_target:
        need = T_target - len(idx)
        extra = np.linspace(0, T_all-1, need+2)[1:-1].round().astype(np.int64)
        idx = np.unique(np.concatenate([idx, extra]))
        idx = idx[:T_target]
    return idx

# ---------------- Main per-video processing ----------------
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Not opened: {video_path}"); return None

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    pose_list=[]; rgb_list=[]; flow_list=[]; score_list=[]
    prev_raw=None; prev_smooth=None; prev_cents=np.empty((0,2),np.float32)
    prev_gray=None; prev_flow=None; prev_sel_cent=None

    idx=0
    with torch.no_grad():
        while True:
            ret, frame_bgr = cap.read()
            if not ret: break

            # Flow features
            gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
            feat_flow, curr_flow, mean_mag = flow_features(prev_gray, gray, prev_flow)
            prev_gray, prev_flow = gray, curr_flow

            # Pose detection
            res = pose_model(frame_bgr, conf=0.35, device=YOLO_DEVICE, half=USE_HALF, verbose=False)
            cur_raw, conf_avg, cur_cents = extract_pose(res, W, H)

            if cur_raw.shape[0]==0:
                # No person: zeros for pose + full-frame RGB embedding
                pose_list.append(np.zeros((TOP_K,117), np.float32))
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                pil = TF.to_pil_image(frame_rgb)
                with torch.amp.autocast('cuda', enabled=(USE_HALF and DEVICE=='cuda')):
                    feat_rgb = rgb_model(rgb_tf(pil).unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu().numpy().astype(np.float32)
                rgb_list.append(feat_rgb)
                flow_list.append(feat_flow)
                # score: low, but use flow magnitude as proxy
                score_list.append(float(mean_mag))
                prev_raw=prev_smooth=None; prev_cents=np.empty((0,2),np.float32); prev_sel_cent=None
                idx+=1
                continue

            # rank persons by confidence and motion
            prev_map = match_prev_curr(prev_cents, cur_cents, max_dist=0.2) if prev_cents.size else [-1]*len(cur_raw)
            mot = np.array([0.0 if (j==-1 or prev_raw is None or j>=len(prev_raw)) else motion_score(cur_raw[i], prev_raw[j])
                            for i,j in enumerate(prev_map)], np.float32)
            scores_p = CONF_W*conf_avg + MOT_W*mot
            order  = np.argsort(scores_p)[::-1][:TOP_K]
            cur_sel_raw   = cur_raw[order]
            cur_sel_cents = cur_cents[order]

            # stable slots 0..K-1
            slot_assign = [-1]*TOP_K; used=set()
            if prev_cents.size:
                for slot in range(min(TOP_K, len(prev_cents))):
                    jbest,dbest=-1,1e9
                    for jj,i in enumerate(order):
                        if jj in used: continue
                        d = np.linalg.norm(prev_cents[slot] - cur_cents[i])
                        if d < dbest and d <= 0.2:
                            dbest, jbest = d, jj
                    if jbest!=-1:
                        slot_assign[slot]=jbest; used.add(jbest)
            for slot in range(TOP_K):
                if slot_assign[slot]==-1:
                    for jj in range(min(TOP_K, len(order))):
                        if jj not in used:
                            slot_assign[slot]=jj; used.add(jj); break

            cur_sel_raw   = np.array([cur_sel_raw[j]   if j!=-1 and j<len(cur_sel_raw) else np.zeros(51,np.float32) for j in slot_assign], np.float32)
            cur_sel_cents = np.array([cur_sel_cents[j] if j!=-1 and j<len(cur_sel_cents) else np.zeros(2,np.float32)  for j in slot_assign], np.float32)

            # align prev
            if prev_raw is None:
                aligned_prev_smooth=[None]*TOP_K; prev_sel_cent=None
            else:
                aligned_prev_smooth=[]; prev_sel_cent=[]
                for k in range(TOP_K):
                    if k < len(prev_smooth):
                        aligned_prev_smooth.append(prev_smooth[k])
                        prev_sel_cent.append(prev_cents[k] if k < len(prev_cents) else cur_sel_cents[k])
                    else:
                        aligned_prev_smooth.append(None)
                        prev_sel_cent.append(cur_sel_cents[k])
                prev_sel_cent = np.array(prev_sel_cent, np.float32)

            # camera compensation
            if prev_sel_cent is not None:
                gdx,gdy = np.mean(cur_sel_cents - prev_sel_cent, axis=0)
            else:
                gdx,gdy = 0.0,0.0

            # smoothing + velocity
            cur_sel_smooth=[]
            for cur, prv in zip(cur_sel_raw, aligned_prev_smooth):
                if prv is None or SMOOTH_ALPHA<=0:
                    cur_sel_smooth.append(cur)
                else:
                    cur_sel_smooth.append((SMOOTH_ALPHA*cur + (1.0-SMOOTH_ALPHA)*prv).astype(np.float32))
            cur_sel_smooth = np.stack(cur_sel_smooth, axis=0) if len(cur_sel_smooth) else np.zeros((TOP_K,51),np.float32)

            feat_102 = [add_velocity_camcomp(cur_s, prv_s, gdx, gdy) for cur_s, prv_s in zip(cur_sel_smooth, aligned_prev_smooth)]
            feat_102 = np.vstack(feat_102)

            # pairwise globals and contact proxies
            pairs=[(0,1),(0,2),(1,2)]
            dists = np.zeros(3,np.float32); speeds=np.zeros(3,np.float32); ious=np.zeros(3,np.float32); apprs=np.zeros(3,np.float32)
            cur_boxes = np.array([bbox_from_kp(kp) for kp in cur_sel_smooth], np.float32)
            for pi,(a,b) in enumerate(pairs):
                dists[pi] = float(np.linalg.norm(cur_sel_cents[a]-cur_sel_cents[b]))
                if prev_sel_cent is not None:
                    dv_a = cur_sel_cents[a] - prev_sel_cent[a]
                    dv_b = cur_sel_cents[b] - prev_sel_cent[b]
                    speeds[pi] = float(np.linalg.norm(dv_a - dv_b))
                    apprs[pi]  = signed_approach(cur_sel_cents[a], cur_sel_cents[b], prev_sel_cent[a], prev_sel_cent[b])
                else:
                    speeds[pi]=0.0; apprs[pi]=0.0
                ious[pi] = iou_xyxy(cur_boxes[a], cur_boxes[b])
            contacts = np.zeros(3,np.float32)
            for pi,(a,b) in enumerate(pairs):
                contacts[pi] = min_wrist_head_dist(cur_sel_smooth[a], cur_sel_smooth[b])

            g15_tile = np.tile(np.hstack([dists,speeds,ious,apprs,contacts]).astype(np.float32)[None,:], (TOP_K,1))
            pose_feat = np.hstack([feat_102, g15_tile])  # (K,117)
            pose_list.append(pose_feat)

            # ROI-RGB embedding on union box
            boxes_norm = []
            for k in range(TOP_K):
                b = bbox_from_kp(cur_sel_smooth[k])
                b = np.clip(b, 0.0, 1.0)
                boxes_norm.append(b)
            boxes_norm = np.stack(boxes_norm, 0)
            roi_norm = union_roi_norm(boxes_norm, take_topn=ROI_TOPN, expand=ROI_EXPAND)
            x1 = int(roi_norm[0] * W); y1 = int(roi_norm[1] * H)
            x2 = int(roi_norm[2] * W); y2 = int(roi_norm[3] * H)
            x1 = max(0, min(x1, W-1)); y1 = max(0, min(y1, H-1))
            x2 = max(x1+1, min(x2, W));  y2 = max(y1+1, min(y2, H))
            roi_bgr = frame_bgr[y1:y2, x1:x2, :]
            roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)
            pil = TF.to_pil_image(roi_rgb)
            with torch.amp.autocast('cuda', enabled=(USE_HALF and DEVICE=='cuda')):
                feat_rgb = rgb_model(rgb_tf(pil).unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu().numpy().astype(np.float32)
            rgb_list.append(feat_rgb)
            flow_list.append(feat_flow)

            # ---- per-frame fight-likeness score (heuristic) ----
            # components: avg motion of top persons, approach+, IoU, inverse contact dist, flow mag proxy, and avg kp conf
            avg_mot = float(np.mean([motion_score(cur_sel_smooth[k], aligned_prev_smooth[k]) if aligned_prev_smooth[k] is not None else 0.0 for k in range(TOP_K)]))
            appr_pos = float(np.maximum(apprs, 0.0).mean())
            iou_mean = float(ious.mean())
            inv_contact = float(1.0/(np.mean(contacts)+1e-6))
            conf_avg = float(np.clip(np.mean(scores_p[order[:TOP_K]]), 0.0, 10.0))
            s = (0.30*avg_mot) + (0.20*appr_pos) + (0.15*iou_mean) + (0.15*inv_contact) + (0.15*mean_mag) + (0.05*conf_avg)
            score_list.append(float(max(0.0, s)))

            # update prev
            prev_raw    = cur_sel_raw.copy()
            prev_smooth = cur_sel_smooth.copy()
            prev_cents  = cur_sel_cents.copy()
            prev_sel_cent = prev_cents.copy()

            idx+=1

    cap.release()

    # Stack all frames
    if len(pose_list)==0:
        return None
    pose_all = np.stack(pose_list, axis=0)          # (T_all, K, 117)
    rgb_all  = np.stack(rgb_list,  axis=0)          # (T_all, 1280)
    flow_all = np.stack(flow_list, axis=0)          # (T_all, 13)
    scores   = np.asarray(score_list, np.float32)   # (T_all,)

    # Select exactly TGT_FRAMES frames by score-aware sampling (chronological order)
    pick = pick_indices_score_aware(scores, TGT_FRAMES)
    pick.sort()
    pose_sel = pose_all[pick]
    rgb_sel  = rgb_all[pick]
    flow_sel = flow_all[pick]

    # If less than TGT_FRAMES, pad zeros at the end (rare: only if len<75)
    T = pose_sel.shape[0]
    if T < TGT_FRAMES:
        pad_p = np.zeros((TGT_FRAMES-T, TOP_K, 117), np.float32)
        pad_r = np.zeros((TGT_FRAMES-T, 1280),      np.float32)
        pad_f = np.zeros((TGT_FRAMES-T, 13),        np.float32)
        pose_sel = np.concatenate([pose_sel, pad_p], axis=0)
        rgb_sel  = np.concatenate([rgb_sel,  pad_r], axis=0)
        flow_sel = np.concatenate([flow_sel, pad_f], axis=0)

    return pose_sel, rgb_sel, flow_sel

def process_folder(src, dst):
    os.makedirs(dst, exist_ok=True)
    vids = [f for f in os.listdir(src) if f.lower().endswith(".mp4")]
    vids.sort()
    for f in vids:
        vp = os.path.join(src, f)
        print(f"▶ Processing: {f}")
        out = process_video(vp)
        if out is None:
            print(f"❌ Skipped (no frames/features): {f}")
            continue
        pose_out, rgb_out, flow_out = out
        sp = os.path.join(dst, f.rsplit(".",1)[0] + ".npz")
        np.savez(sp, pose=pose_out, rgb=rgb_out, flow=flow_out)
        print(f"✅ Saved: {sp} | shapes: pose={pose_out.shape}, rgb={rgb_out.shape}, flow={flow_out.shape}")

def main():
    #  (edit as needed)
    src = r"G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNon_3sec\Fight"
    dst = r"G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight"
    process_folder(src, dst)

if __name__ == "__main__":
    main()


▶ Processing: Violence1_p001.mp4
✅ Saved: G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight\Violence1_p001.npz | shapes: pose=(75, 4, 117), rgb=(75, 1280), flow=(75, 13)
▶ Processing: Violence1_p002.mp4
✅ Saved: G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight\Violence1_p002.npz | shapes: pose=(75, 4, 117), rgb=(75, 1280), flow=(75, 13)
▶ Processing: Violence1_p003.mp4
✅ Saved: G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight\Violence1_p003.npz | shapes: pose=(75, 4, 117), rgb=(75, 1280), flow=(75, 13)
▶ Processing: Violence1_p004.mp4
✅ Saved: G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight\Violence1_p004.npz | shapes: pose=(75, 4, 117), rgb=(75, 1280), flow=(75, 13)
▶ Processing: Violence1_p005.mp4
✅ Saved: G:\Arafat46\Thesis\Dataset\Own_Custom_Data\Home\HomeVioNonPNZ_V2\Fight\Violence1_p005.npz | shapes: pose=(75, 4, 117), rgb=(75, 1280), flow=(75, 13)
▶ Processing: Violence1_p006.mp4
✅ Saved: G:\